In [1]:
from datetime import date

import numpy as np
import pandas as pd

#### Objective: 

Generate a .parquet file with the input Dataset with dates and amount parsed. Dropping unparseable values, and removing columns with no meaning.

#### Depends on:

In [ ]:
f_name = "./data/V2_ENG_interview_dataset.csv"

#### Generates:

In [ ]:
out_f_name = "./data/01-parsed-amounts-and-dates.parquet"

#### General Settings

In [2]:
cols = {
    "cohort": str,
    "agency_number": str,
    "agency_name": str,
    "holder_last_name": str,
    "holder_first_initial": str,
    "amount": str,
    "vendor": str,
    "transaction_date": str,
    "posted_date": str,
    "mcc": str,
    "c1": str,
    "c2": str,
    "c3": str,
    "c4": str,
    "c5": str,
}

# "07/30/2013 12:00:00 AM"
date_format = "%m/%d/%Y %I:%M:%S %p"

use_cols = set(cols.keys()) - set(["c1", "c2", "c3", "c4", "c5"])

-------------------

In [3]:
pd.__version__

'2.1.3'

In [4]:
df = pd.read_csv(f_name, header=0, names=cols.keys(), usecols=use_cols, dtype=cols)

In [5]:
len(df)

442458

In [6]:
df["parsed_amount"] = pd.to_numeric(df["amount"], errors="coerce")

In [7]:
df[df["parsed_amount"].isna()].head(3)

,cohort,agency_number,agency_name,holder_last_name,holder_first_initial,amount,vendor,transaction_date,posted_date,mcc,parsed_amount
473,201307,1000,OKLAHOMA STATE UNIVERSITY,Livsey,K,0 - 500 Co EACH,15,CTC CONSTANTCONTACT.COM,07/26/2013 12:00:00 AM,07/29/2013 12:00:00 AM,NaN
3247,201307,1000,OKLAHOMA STATE UNIVERSITY,Sisco,K,0 - 500 Co EACH|MyLibrary Plus,MyL,25,CTC CONSTANTCONTACT.COM,07/09/2013 12:00:00 AM,NaN
5199,201307,1000,OKLAHOMA STATE UNIVERSITY,Vandiver,C,0 - 1 Published EACH,20,CTC CONSTANTCONTACT.COM,07/17/2013 12:00:00 AM,07/18/2013 12:00:00 AM,NaN


In [8]:
print("parsing errors", len(df[df["parsed_amount"].isna()]))

parsing errors 270


In [9]:
df = df[~df["parsed_amount"].isna()]

In [10]:
len(df)

442188

In [16]:
df["parsed_amount"] = df["parsed_amount"].abs()

In [17]:
# Testing
pd.to_datetime("07/30/2013 12:00:00 AM", format=date_format)

Timestamp('2013-07-30 00:00:00')

In [18]:
df["parsed_transaction_date"] = pd.to_datetime(
    df["transaction_date"],
    format=date_format,
    errors="coerce",
)

df["parsed_posted_date"] = pd.to_datetime(
    df["posted_date"],
    format=date_format,
    errors="coerce",
)

In [19]:
df[df["parsed_transaction_date"].isna()].head(3)

,cohort,agency_number,agency_name,holder_last_name,holder_first_initial,amount,vendor,transaction_date,posted_date,mcc,parsed_amount,parsed_transaction_date,parsed_posted_date
98920,201310,30500,GOVERNOR,Harper,C,670945,4|PARTS EACH,269.55,FIRST CHOICE COFFEE SERVI,10/18/2013 12:00:00 AM,670945.0,NaT,NaT
216596,201309,77000,UNIV. OF OKLA. HEALTH SCIENCES CENTER,CROCKETT,M,670900,|PARTS EACH,595.02,FIRST CHOICE COFFEE SERVI,09/19/2013 12:00:00 AM,670900.0,NaT,NaT
225341,201310,77000,UNIV. OF OKLA. HEALTH SCIENCES CENTER,TOTTEN,S,45096,4|PARTS EACH,360.15,FIRST CHOICE COFFEE SERVI,10/22/2013 12:00:00 AM,45096.0,NaT,NaT


In [20]:
print(
    "parsing errors dates",
    len(df[df["parsed_transaction_date"].isna() | df["parsed_posted_date"].isna()]),
)

parsing errors dates 5


In [21]:
df = df[~(df["parsed_transaction_date"].isna() | df["parsed_posted_date"].isna())]

In [22]:
len(df)

442183

In [23]:
df = df.drop(columns=["posted_date", "transaction_date"])

In [24]:
df = df.rename(
    columns={
        "parsed_transaction_date": "transaction_date",
        "parsed_posted_date": "posted_date",
    }
)

In [25]:
df.to_parquet(out_f_name, index=False)